In [1]:
import pandas as pd
import joblib
rf_model = joblib.load("../artifacts/churn_model.pkl")

feature_names = joblib.load("../artifacts/feature_names.pkl")

df = pd.read_csv("../data/features/customer_segments.csv")

df.head()

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,...,DaySinceLastOrder,CashbackAmount,AverageSpendPerOrder,EngagementScore,LoyaltyLevel,HighValueCustomer,CustomerActivity,CashbackPerOrder,Cluster,BehaviorSegment
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,...,5.0,159.93,79.965,5.0,New,0,2.0,79.965,4,Regular Customers
1,50002,1,9.0,Phone,1,8.0,UPI,Male,3.0,4,...,0.0,120.90,60.450,4.0,Regular,0,1.0,60.450,2,Low Engagement Customers
2,50003,1,9.0,Phone,1,30.0,Debit Card,Male,2.0,4,...,3.0,120.28,60.140,3.0,Regular,0,1.0,60.140,2,Low Engagement Customers
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,...,3.0,134.07,67.035,3.0,New,0,1.0,67.035,4,Regular Customers
4,50005,1,0.0,Phone,1,12.0,CC,Male,3.0,3,...,3.0,129.60,64.800,5.0,New,0,2.0,64.800,2,Low Engagement Customers


In [2]:
#features
X = df.drop(columns=[
    "CustomerID",
    "Churn"
])

X = pd.get_dummies(
    X,
    drop_first=True
)

# Ensure columns match training
X = X.reindex(columns=feature_names, fill_value=0)

print(X.shape)

(5630, 42)


In [3]:
##churn probability
df["ChurnProbability"] = rf_model.predict_proba(X)[:, 1]

df[["CustomerID", "ChurnProbability"]].head()

,CustomerID,ChurnProbability
0,50001,0.970
1,50002,0.990
2,50003,1.000
3,50004,0.995
4,50005,0.995


In [4]:
##assigning risk levels
def risk_level(probability):

    if probability >= 0.80:
        return "High"

    elif probability >= 0.50:
        return "Medium"

    else:
        return "Low"


df["RiskLevel"] = df["ChurnProbability"].apply(risk_level)

df["RiskLevel"].value_counts()

RiskLevel
Low       4672
High       887
Medium      71
Name: count, dtype: int64

In [5]:
##personalized recommendations
def generate_recommendation(row):

    recommendations = []

    if row["Complain"] == 1:
        recommendations.append(
            "Resolve customer complaints through proactive support."
        )

    if row["Tenure"] < 6:
        recommendations.append(
            "Provide a welcome offer or onboarding benefits."
        )

    if row["OrderCount"] < 3:
        recommendations.append(
            "Send personalized product recommendations."
        )

    if row["CashbackAmount"] < 150:
        recommendations.append(
            "Offer cashback rewards or loyalty points."
        )

    if row["RiskLevel"] == "High":
        recommendations.append(
            "Launch an immediate retention campaign."
        )

    if not recommendations:
        recommendations.append(
            "Continue engagement through loyalty programs."
        )

    return " | ".join(recommendations)

In [6]:
##applying recommendations
df["RetentionRecommendation"] = df.apply(
    generate_recommendation,
    axis=1
)

df[[
    "CustomerID",
    "RiskLevel",
    "RetentionRecommendation"
]].head()

,CustomerID,RiskLevel,RetentionRecommendation
0,50001,High,Resolve customer complaints through proactive ...
1,50002,High,Resolve customer complaints through proactive ...
2,50003,High,Resolve customer complaints through proactive ...
3,50004,High,Provide a welcome offer or onboarding benefits...
4,50005,High,Provide a welcome offer or onboarding benefits...


In [7]:
##high risk customers
high_risk = df[df["RiskLevel"] == "High"]

high_risk[[
    "CustomerID",
    "ChurnProbability",
    "RiskLevel",
    "RetentionRecommendation"
]].head(10)

,CustomerID,ChurnProbability,RiskLevel,RetentionRecommendation
0,50001,0.970,High,Resolve customer complaints through proactive ...
1,50002,0.990,High,Resolve customer complaints through proactive ...
2,50003,1.000,High,Resolve customer complaints through proactive ...
3,50004,0.995,High,Provide a welcome offer or onboarding benefits...
4,50005,0.995,High,Provide a welcome offer or onboarding benefits...
5,50006,1.000,High,Resolve customer complaints through proactive ...
6,50007,0.990,High,Send personalized product recommendations. | O...
8,50009,0.975,High,Resolve customer complaints through proactive ...
9,50010,0.955,High,Send personalized product recommendations. | O...
11,50012,1.000,High,Resolve customer complaints through proactive ...


In [ ]:
df.to_csv(
    "../data/features/customer_retention_recommendations.csv",
    index=False
)

print("AI retention recommendations saved successfully!")